# HybridNonlinearFactorGraph

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/hybrid/doc/HybridNonlinearFactorGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --quiet gtsam

`HybridNonlinearFactorGraph` is a `HybridFactorGraph` specifically for representing **nonlinear** hybrid optimization problems. It can contain:

*   `gtsam.NonlinearFactor` (including derived types like `PriorFactor`, `BetweenFactor`)
*   `gtsam.DiscreteFactor` (including `DecisionTreeFactor`, `TableFactor`)
*   `gtsam.HybridNonlinearFactor`

This is the graph type you would typically build to model a system with both continuous states (potentially on manifolds) and discrete modes or decisions, where the relationships are nonlinear.

The primary operation performed on a `HybridNonlinearFactorGraph` is `linearize`, which converts it into a `HybridGaussianFactorGraph` suitable for inference using methods like `eliminateSequential` or `eliminateMultifrontal`.

In [ ]:
import gtsam
import numpy as np
from gtsam import symbol_shorthand

# Factors
from gtsam import HybridNonlinearFactorGraph, HybridNonlinearFactor
from gtsam import PriorFactorPose2, BetweenFactorPose2, Pose2, Point3
from gtsam import DecisionTreeFactor, DiscreteKey

# Values
from gtsam import Values, DiscreteValues, HybridValues

# Results/Intermediate
from gtsam import HybridGaussianFactorGraph

X = symbol_shorthand.X # Continuous Key (Pose2)
D = symbol_shorthand.D # Discrete Key

## Initialization and Adding Factors

In [ ]:
hnfg = gtsam.HybridNonlinearFactorGraph()

# 1. Add a Nonlinear Factor (Prior)
prior_noise = gtsam.noiseModel.Diagonal.Sigmas(Point3(0.1, 0.1, 0.05))
hnfg.add(PriorFactorPose2(X(0), Pose2(0, 0, 0), prior_noise))

# 2. Add a Discrete Factor (Prior on mode)
dk0 = DiscreteKey(D(0), 2) # Binary mode
hnfg.add(DecisionTreeFactor([dk0], "0.8 0.2")) # P(D0=0)=0.8

# 3. Add a HybridNonlinearFactor (Mode-dependent odometry)
# Mode 0 (Grippy): Smaller noise
noise0 = gtsam.noiseModel.Diagonal.Sigmas(Point3(0.1, 0.1, np.radians(1)))
odom0 = BetweenFactorPose2(X(0), X(1), Pose2(1.0, 0, 0), noise0)
# Mode 1 (Slippery): Larger noise
noise1 = gtsam.noiseModel.Diagonal.Sigmas(Point3(0.5, 0.5, np.radians(10)))
odom1 = BetweenFactorPose2(X(0), X(1), Pose2(1.0, 0, 0), noise1)
# Assume equal probability for modes within this factor's context
hybrid_odom = HybridNonlinearFactor(dk0, [odom0, odom1])
hnfg.add(hybrid_odom)

print("HybridNonlinearFactorGraph Contents:")
hnfg.print()

## Error Calculation (`error`, `errorTree`, `discretePosterior`)

Evaluates the combined error (nonlinear factors) and negative log probability (discrete factors) of the graph.

In [ ]:
# Requires HybridValues for evaluation
hybrid_values = gtsam.HybridValues()

# Continuous/Nonlinear Values
nl_vals = gtsam.Values()
nl_vals.insert(X(0), Pose2(0.05, 0.05, np.radians(1))) # Near prior mean
nl_vals.insert(X(1), Pose2(1.0, -0.05, np.radians(0))) # Near odom mean
hybrid_values.insert(nl_vals)

# Discrete Values
disc_vals = gtsam.DiscreteValues()
disc_vals[D(0)] = 0 # Select Grippy mode (lower error)
hybrid_values.insert(disc_vals)

print("\nEvaluating error with HybridValues:")
hybrid_values.print()

# --- error(HybridValues) ---
total_error = hnfg.error(hybrid_values)
print(f"\nTotal graph error: {total_error}")

# --- errorTree(Values) ---
# Calculates error for Nonlinear & HybridNonlinear factors across all discrete modes
# Ignores purely DiscreteFactors for this calculation.
cont_values = hybrid_values.nonlinear()
error_tree = hnfg.errorTree(cont_values)
print("\nError Tree (Nonlinear errors across modes):")
error_tree.print()

# --- discretePosterior(Values) ---
# Calculates P(M | X=x_cont), normalizing exp(-errorTree)
# Includes contributions from purely discrete factors as well.
discrete_posterior = hnfg.discretePosterior(cont_values)
print("\nDiscrete Posterior P(M | X=x):")
discrete_posterior.print()

## Linearization

The primary operation is `linearize`, which converts the graph to a `HybridGaussianFactorGraph` at a given linearization point (continuous `Values`).

In [ ]:
# Linearization point (often the current estimate)
linearization_point = gtsam.Values()
linearization_point.insert(X(0), Pose2(0, 0, 0))
linearization_point.insert(X(1), Pose2(1, 0, 0))

print("\nLinearizing at:")
linearization_point.print()

# Perform linearization
hgfg = hnfg.linearize(linearization_point)

print("\nResulting HybridGaussianFactorGraph:")
hgfg.print()
# Note: NonlinearFactors become JacobianFactors.
#       HybridNonlinearFactors become HybridGaussianFactors.
#       DiscreteFactors remain DiscreteFactors.

## Restriction (`restrict`)

Applies `restrict` to all factors in the graph for a given discrete assignment.

In [ ]:
# Restrict the graph to the 'Grippy' mode (D0=0)
assignment = gtsam.DiscreteValues([(D(0), 0)])
restricted_hnfg = hnfg.restrict(assignment)

print("\nRestricted HybridNonlinearFactorGraph (D0=0):")
restricted_hnfg.print()
# Note: The HybridNonlinearFactor is replaced by its D0=0 component (odom0).
# The DiscreteFactor is removed as its variable is assigned.